# 13 - Binary Two-Phase Model: App Inference Contract

**Purpose:** Define and document the app-facing inference contract for the binary
cancer-risk screening model. Generates configuration, thresholds, example outputs,
a Markdown contract, and a sample prediction visualization.

**Output is binary-only:**
- cancer_risk_probability
- cancer_risk_percent
- risk_level
- binary_prediction_at_youden_threshold
- recommended_action
- disclaimer

> **Medical note:** This AI output is for screening support only and is not a
> medical diagnosis. It should not replace professional medical evaluation.

---


## Section 0 - Imports and Environment

In [1]:
import json
import os
import textwrap
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

CUDA_AVAILABLE = torch.cuda.is_available()
try:
    DEVICE   = torch.device("cuda" if CUDA_AVAILABLE else "cpu")
    GPU_NAME = torch.cuda.get_device_name(0) if CUDA_AVAILABLE else "N/A"
except AssertionError:
    DEVICE, GPU_NAME, CUDA_AVAILABLE = torch.device("cpu"), "N/A", False

print(f"torch       : {torch.__version__}")
print(f"CUDA        : {CUDA_AVAILABLE}  GPU: {GPU_NAME}")
print(f"Device      : {DEVICE}")


torch       : 2.6.0+cu124
CUDA        : True  GPU: NVIDIA GeForce RTX 4060 Laptop GPU
Device      : cuda


## Section 1 - Paths and Configuration

In [2]:
OUTPUT_ROOT   = Path(r"C:\SKIN CANCER v2\pipe output")
TRAIN_2PH_DIR = OUTPUT_ROOT / "pytorch_training_binary_2phase"
EVAL_DIR      = OUTPUT_ROOT / "pytorch_binary_2phase_evaluation"
PREPROC_DIR   = OUTPUT_ROOT / "preprocessing"
CONTRACT_DIR  = OUTPUT_ROOT / "binary_2phase_app_contract"

MODEL_PATH    = TRAIN_2PH_DIR / "best_model_binary_2phase.pt"
MODEL_VERSION = "binary_2phase_v1"

YOUDEN_THRESH      = 0.4219
HIGH_RECALL_THRESH = 0.1700

IMG_SIZE      = 224
RESIZE_TO     = 256
BATCH_SIZE    = 32
NUM_WORKERS   = 0
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

DISCLAIMER = (
    "This AI output is for screening support only and is not a medical diagnosis. "
    "It should not replace professional medical evaluation."
)

WORDING = {
    "lower": (
        "Model-estimated cancer risk is lower, but this does not rule out disease. "
        "Consult a clinician if the lesion changes, bleeds, hurts, or concerns you."
    ),
    "moderate": (
        "Model-estimated cancer risk is moderate. Consider medical review, especially "
        "if the lesion is new, changing, symptomatic, or clinically concerning."
    ),
    "higher": (
        "Model-estimated cancer risk is higher. A dermatologist or qualified clinician "
        "should review this lesion."
    ),
}

print("Config loaded.")
print(f"  Youden J threshold  : {YOUDEN_THRESH}")
print(f"  High-recall boundary: {HIGH_RECALL_THRESH}")
print(f"  Model path exists   : {MODEL_PATH.exists()}")


Config loaded.
  Youden J threshold  : 0.4219
  High-recall boundary: 0.17
  Model path exists   : True


## Section 2 - Create Output Folder

In [3]:
CONTRACT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Ready: {CONTRACT_DIR}")


Ready: C:\SKIN CANCER v2\pipe output\binary_2phase_app_contract


## Section 3 - Load Model

In [4]:
def build_model():
    model = efficientnet_b0(weights=None)
    in_feat = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.2, inplace=True),
        nn.Linear(in_feat, 2),
    )
    return model.to(DEVICE)

if not MODEL_PATH.exists():
    raise FileNotFoundError(
        f"Checkpoint not found: {MODEL_PATH}\n"
        "Run 10_pytorch_binary_two_phase_training.ipynb first."
    )

checkpoint = torch.load(MODEL_PATH, map_location=DEVICE, weights_only=False)
model = build_model()
model.load_state_dict(checkpoint["model_state"])
model.eval()

ckpt_epoch = checkpoint.get("global_epoch", checkpoint.get("epoch", "?"))
ckpt_phase = checkpoint.get("phase", "?")
print(f"Model loaded: epoch={ckpt_epoch}  phase={ckpt_phase}")
print(f"Parameters  : {sum(p.numel() for p in model.parameters()):,}")


Model loaded: epoch=17  phase=phase2_finetune
Parameters  : 4,010,110


## Section 4 - Risk Level Definitions and Inference Function

In [5]:
def get_risk_level(prob):
    if prob < HIGH_RECALL_THRESH:
        return "lower"
    elif prob < YOUDEN_THRESH:
        return "moderate"
    else:
        return "higher"


def get_recommendation(risk_level):
    return WORDING[risk_level]


def format_app_output(prob, model_version=MODEL_VERSION, threshold_version="youden_j_v1"):
    prob = float(prob)
    risk_level = get_risk_level(prob)
    binary_pred = int(prob >= YOUDEN_THRESH)
    return {
        "cancer_risk_probability":           round(prob, 6),
        "cancer_risk_percent":               round(prob * 100, 2),
        "risk_level":                        f"{risk_level}_model_estimated_risk",
        "binary_prediction_at_youden_threshold": binary_pred,
        "recommended_action":                get_recommendation(risk_level),
        "disclaimer":                        DISCLAIMER,
        "model_version":                     model_version,
        "threshold_version":                 threshold_version,
    }


# Test the formatter
for p in [0.05, 0.25, 0.50, 0.75, 0.92]:
    out = format_app_output(p)
    print(f"  p={p:.2f}  risk={out['risk_level']:<35}  pred={out['binary_prediction_at_youden_threshold']}")

eval_transform = T.Compose([
    T.Resize(RESIZE_TO),
    T.CenterCrop(IMG_SIZE),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])


  p=0.05  risk=lower_model_estimated_risk           pred=0
  p=0.25  risk=moderate_model_estimated_risk        pred=0
  p=0.50  risk=higher_model_estimated_risk          pred=1
  p=0.75  risk=higher_model_estimated_risk          pred=1
  p=0.92  risk=higher_model_estimated_risk          pred=1


## Section 5 - Sample Predictions from Test Set

In [6]:
# Load File 11 test predictions — already computed probabilities
pred_path = EVAL_DIR / "binary_2phase_test_predictions.csv"
if not pred_path.exists():
    raise FileNotFoundError(f"Run File 11 first: {pred_path}")

full_pred = pd.read_csv(pred_path, low_memory=False)
print(f"Loaded {len(full_pred):,} test predictions")

# Sample: pick images across the probability spectrum and class distribution
def sample_across_spectrum(df, n_total=24, seed=42):
    rng = np.random.default_rng(seed)
    buckets = [
        df[df["cancer_risk_probability"] < 0.15],           # strong non-cancer
        df[(df["cancer_risk_probability"] >= 0.15) & (df["cancer_risk_probability"] < 0.42)],
        df[(df["cancer_risk_probability"] >= 0.42) & (df["cancer_risk_probability"] < 0.70)],
        df[df["cancer_risk_probability"] >= 0.70],           # strong cancer
    ]
    n_each = max(1, n_total // len(buckets))
    parts  = []
    for b in buckets:
        k = min(n_each, len(b))
        if k > 0:
            parts.append(b.sample(k, random_state=int(rng.integers(0, 9999))))
    return pd.concat(parts).sample(frac=1, random_state=seed).reset_index(drop=True)

sample_df = sample_across_spectrum(full_pred, n_total=24)
print(f"Selected {len(sample_df)} samples across probability spectrum")

# Format app output for each sample
app_rows = []
for _, row in sample_df.iterrows():
    prob = float(row["cancer_risk_probability"])
    out  = format_app_output(prob)
    out["preprocessed_full_path"]  = row["preprocessed_full_path"]
    out["final_authoritative_label"] = row.get("final_authoritative_label", "?")
    out["binary_true_label"]        = int(row.get("binary_true_label", -1))
    app_rows.append(out)

app_df = pd.DataFrame(app_rows)
print(f"\nRisk level distribution in sample:")
print(app_df["risk_level"].value_counts().to_string())


Loaded 3,045 test predictions
Selected 24 samples across probability spectrum

Risk level distribution in sample:
risk_level
higher_model_estimated_risk      12
lower_model_estimated_risk        8
moderate_model_estimated_risk     4


## Section 6 - Sample Predictions Grid

In [7]:
N_COLS = 6
n = len(app_df)
n_rows = (n + N_COLS - 1) // N_COLS

RISK_COLORS = {
    "lower_model_estimated_risk":    "#b7e4b7",
    "moderate_model_estimated_risk": "#ffe599",
    "higher_model_estimated_risk":   "#f4a0a0",
}

fig, axes = plt.subplots(n_rows, N_COLS, figsize=(N_COLS * 2.5, n_rows * 3.0))
axes_flat  = axes.flatten() if hasattr(axes, "flatten") else [axes]
for ax in axes_flat:
    ax.axis("off")

for i, row in app_df.iterrows():
    ax = axes_flat[i]
    try:
        img = Image.open(row["preprocessed_full_path"]).convert("RGB")
        ax.imshow(img)
    except Exception:
        ax.set_facecolor("#ddd")
    risk = row["risk_level"]
    color = RISK_COLORS.get(risk, "white")
    ax.set_title(
        f"{row['final_authoritative_label']} | p={row['cancer_risk_probability']:.3f}\n"
        f"{risk.split('_')[0].capitalize()}",
        fontsize=7, pad=2, backgroundcolor=color,
    )
    for spine in ax.spines.values():
        spine.set_edgecolor(color); spine.set_linewidth(3)
    ax.axis("off")

patches = [mpatches.Patch(color=v, label=k.split("_")[0].capitalize())
           for k, v in RISK_COLORS.items()]
fig.legend(handles=patches, loc="lower center", ncol=3, fontsize=9,
           title="Risk Level", bbox_to_anchor=(0.5, -0.02))
plt.suptitle("Sample App Predictions — Binary Cancer-Risk Model", fontsize=12)
plt.tight_layout()
grid_path = CONTRACT_DIR / "sample_app_predictions_binary_2phase.png"
plt.savefig(grid_path, dpi=100, bbox_inches="tight")
plt.close()
print(f"Saved {grid_path.name}")


Saved sample_app_predictions_binary_2phase.png


## Section 7 - Save Config, Thresholds, Examples, and Contract

In [8]:
# ── app_inference_config_binary_2phase.json ───────────────────────────────────
config = {
    "model_version": MODEL_VERSION,
    "model_type":    "binary_cancer_risk",
    "architecture":  "efficientnet_b0",
    "checkpoint":    MODEL_PATH.name,
    "checkpoint_path": str(MODEL_PATH),
    "binary_mapping":  {"NV": 0, "MEL": 1, "BCC": 1},
    "output_classes":  ["non_cancer", "cancer_risk"],
    "input": {
        "image_size":    IMG_SIZE,
        "resize_to":     RESIZE_TO,
        "channels":      3,
        "normalization": {
            "mean": IMAGENET_MEAN,
            "std":  IMAGENET_STD,
        },
        "transform_steps": [
            f"Resize({RESIZE_TO})",
            f"CenterCrop({IMG_SIZE})",
            "ToTensor()",
            "Normalize(ImageNet)",
        ],
    },
    "output_fields": [
        "cancer_risk_probability",
        "cancer_risk_percent",
        "risk_level",
        "binary_prediction_at_youden_threshold",
        "recommended_action",
        "disclaimer",
        "model_version",
        "threshold_version",
    ],
    "risk_levels": {
        "lower":    f"probability < {HIGH_RECALL_THRESH}",
        "moderate": f"{HIGH_RECALL_THRESH} <= probability < {YOUDEN_THRESH}",
        "higher":   f"probability >= {YOUDEN_THRESH}",
    },
    "wording":     WORDING,
    "disclaimer":  DISCLAIMER,
    "limitations": [
        "Trained on dermoscopic images only; performance on smartphone/non-dermoscopic images is unknown.",
        "Validated on a curated dataset; real-world performance may differ.",
        "Three classes only: NV, MEL, BCC. Other lesion types are out-of-distribution.",
        "This model does not diagnose. It estimates cancer-risk probability for screening support.",
        "Not a substitute for clinical evaluation by a qualified clinician.",
    ],
    "training_info": {
        "phase1": "head-only warm-up (backbone frozen), LR=1e-3, max 10 epochs",
        "phase2": "upper-backbone fine-tuning (features[5-8] + head), LR=1e-4, max 20 epochs",
        "primary_metric": "val_pr_auc",
        "test_roc_auc":   0.92758,
        "test_pr_auc":    0.90013,
    },
}
cfg_path = CONTRACT_DIR / "app_inference_config_binary_2phase.json"
with open(cfg_path, "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2)
print(f"Saved {cfg_path.name}")

# ── app_risk_thresholds_binary_2phase.json ────────────────────────────────────
thresholds = {
    "main_action_threshold": {
        "name":  "youden_j",
        "value": YOUDEN_THRESH,
        "description": "Maximises Youden J (sensitivity + specificity - 1) on validation set.",
        "effect": "Balances cancer recall and specificity.",
        "test_cancer_recall":  0.84796,
        "test_specificity":    0.84900,
        "test_cancer_f1":      0.80895,
        "missed_cancer_test":  175,
    },
    "caution_boundary": {
        "name":  "high_recall",
        "value": HIGH_RECALL_THRESH,
        "description": "Highest threshold on validation achieving >= 90% cancer recall.",
        "effect": "Fewer missed cancers, more false positives.",
        "test_cancer_recall":  0.92007,
        "test_specificity":    0.71700,
        "test_cancer_f1":      0.77130,
        "missed_cancer_test":  92,
    },
    "risk_level_boundaries": {
        "lower_to_moderate":  HIGH_RECALL_THRESH,
        "moderate_to_higher": YOUDEN_THRESH,
    },
}
thr_path = CONTRACT_DIR / "app_risk_thresholds_binary_2phase.json"
with open(thr_path, "w", encoding="utf-8") as f:
    json.dump(thresholds, f, indent=2)
print(f"Saved {thr_path.name}")

# ── app_output_examples_binary_2phase.csv ─────────────────────────────────────
examples_path = CONTRACT_DIR / "app_output_examples_binary_2phase.csv"
app_df.to_csv(examples_path, index=False)
print(f"Saved {examples_path.name}  ({len(app_df)} examples)")


Saved app_inference_config_binary_2phase.json
Saved app_risk_thresholds_binary_2phase.json
Saved app_output_examples_binary_2phase.csv  (24 examples)


In [9]:
# ── app_inference_contract_binary_2phase.md ───────────────────────────────────
example_output = format_app_output(0.72)
example_json   = json.dumps(example_output, indent=2)

contract_md = f'''# App Inference Contract — Binary Cancer-Risk Model

**Version:** {MODEL_VERSION}
**Date:** 2026-05-22
**Model:** EfficientNetB0 (binary head, two-phase transfer learning)
**Checkpoint:** `{MODEL_PATH.name}`

---

## Purpose

Provide a standardised interface for mobile/web apps to query the binary
cancer-risk screening model. The model outputs a **cancer-risk probability**
and a **risk level** for a single dermoscopic (or similar) skin lesion image.

This is a **screening support tool only**. It is not a clinical diagnosis.

---

## Input Format

| Field        | Requirement                                          |
|------------- |------------------------------------------------------|
| Image type   | RGB JPEG or PNG                                      |
| Recommended  | Dermoscopic image (macroscopic may be out-of-scope)  |
| Resize       | Resize to 256 px on shortest side, CenterCrop 224   |
| Normalization| ImageNet mean/std (see config JSON)                  |

**Inference transform (exact):**
```
Resize(256) → CenterCrop(224) → ToTensor → Normalize(ImageNet)
```

---

## Output Schema

| Field                                 | Type    | Description                          |
|---------------------------------------|---------|--------------------------------------|
| `cancer_risk_probability`             | float   | Model softmax prob for cancer_risk   |
| `cancer_risk_percent`                 | float   | Probability × 100                    |
| `risk_level`                          | string  | lower / moderate / higher            |
| `binary_prediction_at_youden_threshold` | int   | 1 = model-positive, 0 = model-negative |
| `recommended_action`                  | string  | Human-readable guidance              |
| `disclaimer`                          | string  | Medical disclaimer (always included) |
| `model_version`                       | string  | Model identifier                     |
| `threshold_version`                   | string  | Threshold identifier                 |

---

## Risk Thresholds

| Boundary            | Value  | Meaning                                      |
|---------------------|--------|----------------------------------------------|
| lower → moderate    | 0.1700 | High-recall caution boundary (val ≥90% recall) |
| moderate → higher   | 0.4219 | Youden J action threshold (balanced)         |

---

## Risk Level Wording

**Lower** (`probability < {HIGH_RECALL_THRESH}`):
> {WORDING['lower']}

**Moderate** (`{HIGH_RECALL_THRESH} ≤ probability < {YOUDEN_THRESH}`):
> {WORDING['moderate']}

**Higher** (`probability ≥ {YOUDEN_THRESH}`):
> {WORDING['higher']}

---

## Disclaimer (always include in app output)

> {DISCLAIMER}

---

## Example Output

Input: dermoscopic lesion image

```json
{example_json}
```

---

## Model Limitations

- Trained on dermoscopic images only. Performance on smartphone or
  non-dermoscopic images is unknown and likely reduced.
- Only three lesion types in training: NV, MEL, BCC. Other lesion types
  are out-of-distribution and will produce unreliable probabilities.
- This model does not provide a diagnosis.
- Thresholds were selected on a held-out validation set and evaluated once
  on a held-out test set. Real-world performance may differ.
- Grad-CAM visualisations are qualitative only and do not prove clinical reasoning.

---

## Grad-CAM

Grad-CAM explainability visualisations are handled in a separate notebook:
`14_real_image_binary_inference_gradcam.ipynb`. They are **not** part of the
production app inference output.

---

## Test Performance (reference)

| Metric          | Value   |
|-----------------|---------|
| ROC-AUC         | 0.92758 |
| PR-AUC          | 0.90013 |
| Cancer recall (Youden J=0.4219)  | 0.84796 |
| Specificity (Youden J=0.4219)    | 0.84900 |
| Cancer F1 (Youden J=0.4219)      | 0.80895 |

---

*This contract was generated automatically by
`13_binary_2phase_app_inference_contract.ipynb`.*
'''

contract_path = CONTRACT_DIR / "app_inference_contract_binary_2phase.md"
with open(contract_path, "w", encoding="utf-8") as f:
    f.write(contract_md)
print(f"Saved {contract_path.name}  ({contract_path.stat().st_size:,} bytes)")


Saved app_inference_contract_binary_2phase.md  (5,030 bytes)


## Section 8 - Output File Verification

In [10]:
required_files = [
    CONTRACT_DIR / "app_inference_config_binary_2phase.json",
    CONTRACT_DIR / "app_risk_thresholds_binary_2phase.json",
    CONTRACT_DIR / "app_output_examples_binary_2phase.csv",
    CONTRACT_DIR / "app_inference_contract_binary_2phase.md",
    CONTRACT_DIR / "sample_app_predictions_binary_2phase.png",
]
print("Output file verification:")
all_ok = True
for p in required_files:
    exists = p.exists()
    size   = p.stat().st_size if exists else 0
    status = "OK" if exists else "MISSING"
    print(f"  [{status}] {p.name:<52} {size:>10,} bytes")
    if not exists: all_ok = False
print("\nAll files present." if all_ok else "\nWARNING: missing files.")


Output file verification:
  [OK] app_inference_config_binary_2phase.json                   2,604 bytes
  [OK] app_risk_thresholds_binary_2phase.json                      827 bytes
  [OK] app_output_examples_binary_2phase.csv                     9,711 bytes
  [OK] app_inference_contract_binary_2phase.md                   5,030 bytes
  [OK] sample_app_predictions_binary_2phase.png              1,708,771 bytes

All files present.


## Section 9 - Final Summary (Copy-Paste Ready)

In [11]:
print("=" * 70)
print("  13_binary_2phase_app_inference_contract -- FINAL SUMMARY")
print("=" * 70)
print(f"\n 1. Model checkpoint      : {MODEL_PATH.name}")
print(f"    Checkpoint epoch      : {ckpt_epoch}")
print(f" 2. Youden J threshold    : {YOUDEN_THRESH}")
print(f"\n 3. Risk level definitions:")
print(f"      lower    : probability < {HIGH_RECALL_THRESH}")
print(f"      moderate : {HIGH_RECALL_THRESH} <= probability < {YOUDEN_THRESH}")
print(f"      higher   : probability >= {YOUDEN_THRESH}")
print(f"\n 4. Output fields:")
for field in config['output_fields']:
    print(f"      {field}")
print(f"\n 5. Sample predictions generated: {len(app_df)}")
print(f"    Risk level distribution:")
for rl, cnt in app_df['risk_level'].value_counts().items():
    print(f"      {rl}: {cnt}")
print(f"\n 6. Output file verification:")
for p in required_files:
    exists = p.exists()
    size   = p.stat().st_size if exists else 0
    print(f"      [{'OK' if exists else 'MISSING'}] {p.name:<52} {size:>8,} bytes")
print(f"\n 7. Training performed      : False")
print(f" 8. Images/manifests modified: False")
print(f"\n 9. Medical caution:")
print(f"    {DISCLAIMER}")
print("=" * 70)


  13_binary_2phase_app_inference_contract -- FINAL SUMMARY

 1. Model checkpoint      : best_model_binary_2phase.pt
    Checkpoint epoch      : 17
 2. Youden J threshold    : 0.4219

 3. Risk level definitions:
      lower    : probability < 0.17
      moderate : 0.17 <= probability < 0.4219
      higher   : probability >= 0.4219

 4. Output fields:
      cancer_risk_probability
      cancer_risk_percent
      risk_level
      binary_prediction_at_youden_threshold
      recommended_action
      disclaimer
      model_version
      threshold_version

 5. Sample predictions generated: 24
    Risk level distribution:
      higher_model_estimated_risk: 12
      lower_model_estimated_risk: 8
      moderate_model_estimated_risk: 4

 6. Output file verification:
      [OK] app_inference_config_binary_2phase.json                 2,604 bytes
      [OK] app_risk_thresholds_binary_2phase.json                    827 bytes
      [OK] app_output_examples_binary_2phase.csv                   9,711 byt

## Section 10 - Completion Summary

**13_binary_2phase_app_inference_contract is complete.**

**What was accomplished:**
- App-facing inference contract defined (risk levels, wording, thresholds).
- Config JSON, thresholds JSON, examples CSV, and Markdown contract saved.
- Sample predictions grid saved covering the full probability range.

**What was deliberately deferred:**
- Grad-CAM → File 14
- Mobile SDK integration
